# ATLAS Traffic — ajuste de YOLO a la esquina (fine-tuning)

Entrena `yolov8s` con imágenes de **Av. Córdoba y Rodríguez Peña**, para que aprenda cómo se
ven acá los autos, motos, colectivos, camiones y **personas**: chicos, comprimidos y desde un
sexto piso.

**Antes de empezar:** menú **Entorno de ejecución → Cambiar tipo de entorno de ejecución →
GPU (T4)** → Guardar. Después **Entorno de ejecución → Ejecutar todas**.

Tarda unos 15–30 minutos. Al final se descarga el archivo `atlas_esquina_yolov8s.pt`:
copialo a `atlas-traffic/modelos/` en tu PC y avisale a Claude para que lo evalúe.

In [ ]:
# 1. Verificar que haya GPU (tiene que aparecer una Tesla T4 o similar)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Bajar el proyecto (incluye el dataset) e instalar Ultralytics
!git clone -q https://github.com/licgermancardenas-crypto/ATLAS-TRAFFIC.git
!pip -q install ultralytics

In [ ]:
# 3. Descomprimir el dataset y apuntar data.yaml a su carpeta
import pathlib
import zipfile

import yaml

destino = pathlib.Path("/content/dataset")
zipfile.ZipFile("ATLAS-TRAFFIC/entrenamiento/dataset_esquina.zip").extractall(destino)
data = yaml.safe_load(open(destino / "data.yaml"))
data["path"] = str(destino)
yaml.safe_dump(data, open(destino / "data.yaml", "w"))
for parte in ("train", "val"):
    print(parte, len(list((destino / "images" / parte).glob("*.jpg"))), "imágenes")
print("clases:", data["names"])

In [ ]:
# 4. Entrenar. Parte del yolov8s general y lo ajusta a esta esquina.
#    patience=20: si 20 épocas seguidas no mejora, corta antes.
from ultralytics import YOLO

modelo = YOLO("yolov8s.pt")
modelo.train(
    data="/content/dataset/data.yaml",
    imgsz=640,  # la misma resolución que usa el sistema en el día a día
    epochs=80,
    patience=20,
    batch=16,
    seed=0,
    project="/content/corridas",
    name="esquina",
    exist_ok=True,
)

In [ ]:
# 5. Resultado sobre las imágenes de validación (que el modelo no usó para entrenar)
mejor = YOLO("/content/corridas/esquina/weights/best.pt")
m = mejor.val(data="/content/dataset/data.yaml", imgsz=640, verbose=False)
print(f"mAP50 general: {m.box.map50:.3f}")
for i, nombre in mejor.names.items():
    print(f"  {nombre:<11} mAP50 = {m.box.maps[i]:.3f}")

**Cómo leer el resultado:** mAP50 va de 0 a 1 y mide qué tan bien encuentra cada clase en
imágenes que no vio. Ojo: las etiquetas de validación también son automáticas (del "modelo
maestro"), así que esto mide cuánto aprendió a imitar al maestro. La prueba que importa la
hace Claude después, contra los conteos manuales.

In [ ]:
# 6. Descargar el modelo entrenado
import shutil

from google.colab import files

shutil.copy("/content/corridas/esquina/weights/best.pt", "/content/atlas_esquina_yolov8s.pt")
files.download("/content/atlas_esquina_yolov8s.pt")